# YOLOv8 Object Detection Pipeline — COCO128

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ares-infenus/yolov8-object-detection-pipeline/blob/main/notebooks/training_pipeline.ipynb)

Complete training, evaluation, inference, and export pipeline using YOLOv8n on COCO128.

**Requirements**: Google Colab with GPU runtime (Runtime > Change runtime type > T4 GPU)

## Phase 0: Environment Setup

In [ ]:
# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/proyecto_yolov8'
import os
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f'Project dir: {DRIVE_PROJECT}')

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python-headless onnx onnxruntime matplotlib seaborn pyyaml

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Clone repo (if running from Colab)
!git clone https://github.com/Ares-infenus/yolov8-object-detection-pipeline.git 2>/dev/null || echo 'Repo already cloned'
%cd yolov8-object-detection-pipeline

# Run setup verifier
!python scripts/fase0_setup_verificar.py

## Phase 1: Data Preparation

In [ ]:
# Download COCO128 dataset
!python scripts/download_dataset.py

# Download pre-trained weights
!python scripts/download_weights.py

# Download test video
!python scripts/download_test_video.py

# Verify data
!python scripts/fase1_datos_verificar.py

## Phase 2: Training

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8n
model = YOLO('models/yolov8n.pt')

# Train on COCO128
results = model.train(
    data='coco128.yaml',
    epochs=50,
    imgsz=640,
    batch=16,       # Reduce to 8 if OOM
    name='train',
    exist_ok=True,
)

In [ ]:
# Verify training
!python scripts/fase2_entrenamiento_verificar.py

# Save checkpoint to Drive
import shutil
from pathlib import Path

# Find the actual training output directory
train_dir = None
for candidate in sorted(Path('.').rglob('best.pt')):
    train_dir = candidate.parent.parent  # weights/best.pt -> train dir
    break

if train_dir and train_dir.exists():
    dst = f'{DRIVE_PROJECT}/runs/detect/train'
    shutil.copytree(str(train_dir), dst, dirs_exist_ok=True)
    print(f'Checkpoint saved to Drive from: {train_dir}')
else:
    print('⚠️  No training output found yet. Check that Phase 2 training completed.')

## Phase 3: Evaluation

In [ ]:
!python src/evaluate.py
!python scripts/fase3_evaluacion_verificar.py

## Phase 4: Inference + Speed Benchmark

In [ ]:
!python src/inference.py
!python scripts/fase4_inferencia_verificar.py

In [ ]:
# Display sample detections
from IPython.display import Image, display
from pathlib import Path

for img in sorted(Path('results/samples').glob('*.jpg')):
    print(f'\n{img.name}')
    display(Image(filename=str(img), width=500))

## Phase 5: ONNX Export

In [ ]:
!python src/export.py
!python scripts/fase5_exportacion_verificar.py

## Phase 6: Demo Video

In [ ]:
!python src/demo_video.py
!python scripts/fase6_demo_verificar.py

## Generate Executive Report

In [ ]:
!python scripts/generate_executive_report.py

## General Health Check

In [ ]:
!python scripts/comprobador_general.py

## Save All Results to Drive

In [ ]:
import shutil

# Save everything to Drive
for src_dir in ['results', 'docs/images', 'models']:
    dst = f'{DRIVE_PROJECT}/{src_dir}'
    if Path(src_dir).exists():
        shutil.copytree(src_dir, dst, dirs_exist_ok=True)
        print(f'Saved {src_dir} -> {dst}')

print('\nAll results saved to Google Drive!')